In [ ]:
from pgp_vlm_compression import TinyCLIP


# model, _, preprocess = TinyCLIP.create_model_and_transforms("TinyCLIP-ViT-8M-16-Text-3M", pretrained='YFCC15M')
# model, _, preprocess = TinyCLIP.create_model_and_transforms("TinyCLIP-auto-ViT-22M-32-Text-10M", pretrained='LAION400M')
model, _, preprocess = TinyCLIP.create_model_and_transforms("TinyCLIP-auto-ViT-63M-32-Text-31M", pretrained='LAIONYFCC400M')

# count the number of parameters in millions
total_params = sum(p.numel() for p in model.parameters()) / 1_000_000
print(f'Total number of parameters: {total_params:.2f}M')

Total number of parameters: 120.03M


In [ ]:
from pgp_vlm_compression.prefiltering.prefilter import prefilter_image
from pgp_vlm_compression import TinyCLIP
from PIL import Image
import torch



arch = "TinyCLIP-auto-ViT-22M-32-Text-10M"
pretrained = 'LAION400M'
model, _, preprocess = TinyCLIP.create_model_and_transforms(arch, pretrained)
model = model.cuda()
clip_tokenizer = TinyCLIP.get_tokenizer(arch)

image_file = '03-Confusing-Pictures.jpg'
image = Image.open(image_file).convert('RGB')
# prompt = "What make is the yellow cab in the image?"
# prompt = "What make is the yellow SUV in the image?"
# prompt = "What is the man ironing in the image?"
prompt = "What is it funny here?"

device = torch.device("cuda", 0)
torch.cuda.reset_peak_memory_stats(device)
before = torch.cuda.memory_allocated(device)

prefiltered_image = prefilter_image(
    clip_model=model,
    clip_tokenizer=clip_tokenizer,
    clip_preprocess=preprocess,
    image=image,
    prompt=prompt,
    logit_scale=200.0,
    sigma_min=0.1,
    sigma_max=10.0,
    method="exponential",
    ksize=19,
    strict_text_summ=False,
    target_num_tiles=24,
    preserve_size=True,
    pre_text_summ=True,
    preproc_prompt=True,
)

torch.cuda.synchronize(device)
after = torch.cuda.memory_allocated(device)
peak = torch.cuda.max_memory_allocated(device)

print(f"Δ allocated: {(after - before)/1024**2:.2f} MiB")
print(f"Peak allocated (during block): {peak/1024**2:.2f} MiB")

# prefiltered_image.save("prefiltered_iron.jpg")
# prefiltered_image.show()

Time taken: 600.36 ms
Δ allocated: 8.12 MiB
Peak allocated (during block): 305.24 MiB
